In [ ]:
import sys
import os
from pathlib import Path

# Add project root to path (adjust if notebook is in a subfolder)
project_root = Path.cwd().parent  # if notebook is in experiments/ or similar
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


import warnings

# Suppress the specific future warning from torchrl
warnings.filterwarnings(
    "ignore", 
    category=FutureWarning, 
    module="torchrl.modules.mcts.scores"
)

import torch
from torchrl.envs import EnvBase
from torchrl.data import (
    Composite, 
    Unbounded, 
    Bounded,
    Stacked
    
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import numpy as np

from urbanmarl.envs.urbanmarl_env import UrbanMARLEnv

In [ ]:
np.sqrt(2)

In [ ]:
velocity = 50.0 / np.sqrt(2)
velocity = 50.0 / 1.732
min_vel_v = np.array([-1, -1, -1])
max_vel_v = np.array([1, 1, 1])


np.linalg.norm(min_vel_v), np.linalg.norm(max_vel_v), np.linalg.norm(min_vel_v) * velocity, np.linalg.norm(max_vel_v) * velocity

In [ ]:
vec = np.array([1, 1, 1]) - np.array([-1, -1, -1])
velocity = 50.0 #* np.sqrt(2)

min_vel_v = - velocity * vec * np.linalg.norm(vec)
max_vel_v = velocity * vec #* np.linalg.norm(vec)
vec, velocity, velocity * vec, min_vel_v, max_vel_v, np.linalg.norm(max_vel_v)

In [ ]:
batch_size = 8 
n_uavs = 3
target_shape = torch.Size([batch_size, n_uavs, 7])
low_bound = torch.tensor([-1, -1, 0, -1, -1, -1, 0], dtype=torch.float32).expand(target_shape)
high_bound = torch.tensor([1, 1, 1, 1, 1, 1, 1], dtype=torch.float32).expand(target_shape)
observation_spec = Composite(
    agents = Composite(
        observation = Bounded(
            low = low_bound,
            high = high_bound,
            shape=target_shape,
            dtype=torch.float32
        ),
        shape=torch.Size([batch_size, n_uavs])
    ),
    shape = torch.Size([batch_size]),
    device=device
)

# Or
area_size = (500, 500)
min_x, min_y = - area_size[0] / 2, area_size[1] / 2
max_x, max_y = area_size[0] / 2, area_size[1] / 2
min_z, max_z = 0.0, 200.0
velocity = 50.0  # m/s
min_energy, max_energy = 0, 100 # W
# (x, y, z, v_x, v_y, v_z, energy, qu
low_bound = torch.tensor([min_x, min_x, min_z, -velocity, -velocity, -velocity, min_energy], dtype=torch.float32).expand(target_shape)
high_bound = torch.tensor([max_x, max_y, max_z, velocity, velocity, velocity, max_energy], dtype=torch.float32).expand(target_shape)
observation_spec = Composite(
    agents = Composite(
        observation = Bounded(
            low = low_bound,
            high = high_bound,
            shape=target_shape,
            dtype=torch.float32
        ),
        shape=torch.Size([batch_size, n_uavs])
    ),
    shape = torch.Size([batch_size]),
    device=device
)

# Or
observation_spec = Composite(
    agents = Composite(
        observation = Composite(
            position = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
            velocity = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
            energy = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
            queue_length = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
            shape=torch.Size([batch_size, n_uavs])
        ),
        shape=torch.Size([batch_size, n_uavs])
    ),
    shape = torch.Size([batch_size]),
    device=device
)

In [ ]:
config = {
    "num_uavs": 3,
    "num_ues": 20,
    "area_size": (500, 500),
    "max_time_slots": 50,
    "max_horizontal_speed": 49.0,
    "max_vertical_speed": 12.0,
    "max_transmit_power": 5.0,
    "frequency_ghz": 29.0,
    "g2a_bandwidth": 10e6,
    "noise_figure_db": 7.0
}

environment = UrbanMARLEnv(config=config, device=device, batch_size=torch.Size([2]))

In [ ]:
environment.reset()

In [ ]:
environment.map_engine.plot()

In [ ]:
batch_size = 8 
n_uavs = 3
target_shape = torch.Size([batch_size, n_uavs, 8])
low_bound = torch.tensor([-1, -1, -1, -1, -1, -1, 0, 0], dtype=torch.float32).expand(target_shape)
high_bound = torch.tensor([1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.float32).expand(target_shape)
observation_spec = Composite(
    agents = Composite(
        observation = Bounded(
            low = low_bound,
            high = high_bound,
            # position = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
            # velocity = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
            # energy = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
            # queue_length = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
            shape=target_shape,
            dtype=torch.float32
        ),
        shape=torch.Size([batch_size, n_uavs])
    ),
    shape = torch.Size([batch_size]),
    device=device
)

observation_spec.shape, len(observation_spec)

In [ ]:
batch_size = 8
n_uavs = 3
device = "cpu"

# 1. Define base 1D feature bounds
base_low = torch.tensor([-1, -1, -1, -1, -1, -1, 0, 0], dtype=torch.float32, device=device)
base_high = torch.tensor([1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.float32, device=device)

# 2. Target explicit leaf shape
target_shape = torch.Size([batch_size, n_uavs, 8])

# 3. Expand the bounds so their non-singleton shapes match target_shape exactly
low_bound = base_low.expand(target_shape)
high_bound = base_high.expand(target_shape)

observation_spec = Composite(
    agents = Composite(
        observation = Bounded(
            low = low_bound,
            high = high_bound,
            shape = target_shape,  # Exact match with low and high shapes
            dtype = torch.float32,
            device = device
        ),
        shape = torch.Size([batch_size, n_uavs])
    ),
    shape = torch.Size([batch_size]),
    device = device
)
observation_spec.shape, len(observation_spec), observation_spec

In [ ]:
observation_spec['agents', 'observation'].sample()

In [ ]:
batch_size = 8 
n_uavs = 3
target_shape = torch.Size([batch_size, n_uavs, 8])
low_bound = torch.tensor([-1, -1, -1, -1, -1, -1, 0, 0], dtype=torch.float32).expand(target_shape)
high_bound = torch.tensor([1, 1, 1, 1, 1, 1, 1, 1], dtype=torch.float32).expand(target_shape)
observation_spec = Composite(
    agents = Composite(
        observation = Bounded(
            low = low_bound,
            high = high_bound,
            # position = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
            # velocity = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
            # energy = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
            # queue_length = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
            shape=target_shape,
            dtype=torch.float32
        ),
        shape=torch.Size([batch_size, n_uavs])
    ),
    shape = torch.Size([batch_size]),
    device=device
)

observation_spec.shape, len(observation_spec)

In [ ]:
observation_spec['agents','observation'].sample().shape

In [ ]:
sample = observation_spec.sample()
sample['agents','observation']

In [ ]:
batch_size = 8 #torch.Size([8])
n_uavs = 3
observation_spec = Composite(
    agents = Composite(
        observation = Unbounded(shape=torch.Size([batch_size, n_uavs, 8])),
        # observation = Composite(
        #     position = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
        #     velocity = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
        #     energy = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
        #     queue_length = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
        #     shape=torch.Size([batch_size, n_uavs])
        # ),
        shape=torch.Size([batch_size, n_uavs])
    ),
    shape = torch.Size([batch_size]),
    device=device
)

observation_spec.shape, observation_spec['agents','observation'].sample()

In [ ]:
batch_size = 8 #torch.Size([8])
n_uavs = 3
observation_spec = Composite(
    agents = Composite(
        observation = Composite(
            position = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
            velocity = Unbounded(shape=torch.Size([batch_size, n_uavs, 3])),
            energy = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
            queue_length = Unbounded(shape=torch.Size([batch_size, n_uavs, 1])),
            shape=torch.Size([batch_size, n_uavs])
        ),
        shape=torch.Size([batch_size, n_uavs])
    ),
    shape = torch.Size([batch_size]),
    device=device
)

observation_spec.shape, observation_spec['agents','observation'].sample().shape

In [ ]:
print(observation_spec.rand())

In [ ]:
obs = observation_spec["agents"]

In [ ]:
obs.sample()

In [ ]:
observation_spec["agents"]['observation']['position'].sample()
observation_spec["agents"]['observation'].sample()

In [ ]:
s = obs.sample()
s.size()

In [ ]:
observation_spec["agents"]['observation'].shape